# OCR A Level Computer Science: Binary Number Formats and Arithmetic

### Slideshow lesson recap (based on the two original PowerPoints)
- Binary arithmetic and overflow
- Sign and magnitude
- Two's complement
- Floating point representation and normalisation

This notebook is designed for teaching and worked examples with Manim animations.

## Lesson objectives
By the end of this recap, students should be able to:
1. Represent negative integers using sign and magnitude and two's complement.
2. Add and subtract binary values, including using two's complement for subtraction.
3. Explain overflow in fixed-width binary arithmetic.
4. Convert between denary and floating point binary forms.
5. Normalise floating point numbers with positive and negative mantissas.

In [1]:
# If needed in a fresh environment, uncomment this line and run once.
#%pip install manim

def to_bin(n: int, bits: int = 8) -> str:
    return format(n & ((1 << bits) - 1), f"0{bits}b")

def sign_magnitude(value: int, bits: int = 8) -> str:
    if bits < 2:
        raise ValueError("bits must be at least 2")
    sign = 0 if value >= 0 else 1
    magnitude = abs(value)
    if magnitude > (2 ** (bits - 1) - 1):
        raise ValueError("value out of range for sign and magnitude")
    return str(sign) + format(magnitude, f"0{bits - 1}b")

def twos_complement(value: int, bits: int = 8) -> str:
    min_val = -(2 ** (bits - 1))
    max_val = 2 ** (bits - 1) - 1
    if not (min_val <= value <= max_val):
        raise ValueError("value out of range for two's complement")
    return to_bin(value, bits)

print("Helpers ready.")

Helpers ready.


## Binary addition and overflow recap
Binary addition follows the same carry logic as denary.

Rules:
- 0 + 0 = 0
- 0 + 1 = 1
- 1 + 1 = 10 (sum 0, carry 1)
- 1 + 1 + 1 = 11 (sum 1, carry 1)

In fixed width arithmetic (for example, 8 bits), extra carry out of the leftmost bit is discarded.

In [2]:
# Worked example: 8-bit overflow
a, b, bits = 200, 100, 8
raw_sum = a + b
wrapped = raw_sum & ((1 << bits) - 1)

print(f"a = {a:3d} -> {to_bin(a, bits)}")
print(f"b = {b:3d} -> {to_bin(b, bits)}")
print(f"raw sum = {raw_sum} -> {format(raw_sum, '09b')} (needs 9 bits)")
print(f"stored in 8 bits -> {to_bin(wrapped, bits)} = {wrapped}")
print("Overflow occurred because the true sum does not fit in 8 bits.")

a = 200 -> 11001000
b = 100 -> 01100100
raw sum = 300 -> 100101100 (needs 9 bits)
stored in 8 bits -> 00101100 = 44
Overflow occurred because the true sum does not fit in 8 bits.


## Sign and magnitude
- Leftmost bit is the sign (0 positive, 1 negative).
- Remaining bits store the magnitude.

Example in 8 bits:
- +3 = 00000011
- -3 = 10000011

Issue: ordinary binary addition does not behave nicely with sign and magnitude.

In [3]:
# Worked example from the lesson: (+3) + (-3) in sign and magnitude
pos3 = int(sign_magnitude(3, 8), 2)
neg3 = int(sign_magnitude(-3, 8), 2)
sum_bits = to_bin(pos3 + neg3, 8)

print(f"+3 (sign-mag): {sign_magnitude(3, 8)}")
print(f"-3 (sign-mag): {sign_magnitude(-3, 8)}")
print(f"binary add result: {sum_bits}")
print("This is not 0, showing why sign and magnitude is awkward for arithmetic hardware.")

+3 (sign-mag): 00000011
-3 (sign-mag): 10000011
binary add result: 10000110
This is not 0, showing why sign and magnitude is awkward for arithmetic hardware.


## Two's complement
Two's complement is the standard way to represent signed integers in hardware.

To get -x from +x:
1. Write +x in binary
2. Flip all bits (one's complement)
3. Add 1

Range for n bits: $-(2^{n-1})$ to $2^{n-1} - 1$

In [4]:
# Worked examples: conversion and subtraction using addition
bits = 8
value = -43
print(f"-43 in 8-bit two's complement: {twos_complement(value, bits)}")

# 65 - 43 is done as 65 + (-43)
a = 65
b = -43
result = (a + b) & ((1 << bits) - 1)

print(f"65   -> {twos_complement(65, bits)}")
print(f"-43  -> {twos_complement(-43, bits)}")
print(f"sum  -> {to_bin(result, bits)} = {result}")
print("Ignoring carry out gives the correct answer: 22.")

-43 in 8-bit two's complement: 11010101
65   -> 01000001
-43  -> 11010101
sum  -> 00010110 = 22
Ignoring carry out gives the correct answer: 22.


In [5]:
# Manim notebook bootstrap: robust on Windows mapped-drive/UNC paths.

from pathlib import Path

import os



ip = get_ipython()



# Use one canonical working path (often UNC on Windows) so Manim path comparisons succeed.

os.chdir(str(Path.cwd().resolve()))



# Register %%manim cell magic directly when needed.

if "manim" not in ip.magics_manager.magics.get("cell", {}):

    from manim.utils.ipython_magic import ManimMagic

    ip.register_magics(ManimMagic)



from manim import config

config["media_dir"] = str(Path.cwd() / "media")



# Embed rendered videos in notebook output to avoid UNC/local file loading issues.

config["media_embed"] = True



print("Manim magic ready. media_dir:", config["media_dir"])

print("Manim media_embed:", config["media_embed"])

Manim magic ready. media_dir: /workspaces/OCR-A-Level-Computing-Lessons/media
Manim media_embed: True


## Manim Troubleshooting (Teacher Notes)

If an animation cell fails, check these in order:

1. Run the setup code cell immediately above this slide first.
2. Confirm you are using the correct notebook kernel (the one with `manim` installed).
3. If you see: "The manim module is not an IPython extension", ignore it if the setup cell says `Manim magic ready`.
4. If rendering fails with path/subpath errors on Windows mapped drives, re-run the setup cell (it normalises to a canonical path).
5. Re-run the animation cell. If needed, restart the kernel and run from the top.

Expected success signal:
- You should see `Manim Community v...` and then an embedded video output.

Tip for Codespaces:
- Use this notebook as-is; it already sets `media_dir` to a local `media` folder for predictable outputs.

If render logs show success but video does not play, this is usually a notebook viewer autoplay/policy restriction rather than a Manim render failure.

In [6]:
%%manim -qm SignMagnitudeDemo

from manim import *


class SignMagnitudeDemo(Scene):

    def construct(self):
        title = Text("Sign and Magnitude", font_size=40).to_edge(UP)
        self.play(Write(title))

        s1 = VGroup(
            Text("Sign and magnitude example", font_size=30, color=BLUE),
            Text("+3  = 00000011", font_size=28),
            Text("-3  = 10000011", font_size=28),
            Text("Add: 00000011 + 10000011 = 10000110", font_size=28),
            Text("This represents -6, not 0", font_size=28, color=RED),
            Text("Sign and magnitude breaks ordinary addition", font_size=26, color=YELLOW),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.3).next_to(title, DOWN, buff=0.45).to_edge(LEFT, buff=0.8)

        for line in s1:
            self.play(FadeIn(line, shift=UP * 0.1), run_time=0.55)
            self.wait(0.2)

        wrong_box = SurroundingRectangle(s1[4], color=RED, buff=0.1)
        self.play(Create(wrong_box))
        self.wait(20)


Manim Community v0.20.1

[05/18/26 14:09:41] INFO     Animation 0 : Using cached data (hash :                           ]8;id=5599393;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599394;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             1987512680_3797058306_223132457)                                                      

                    INFO     Animation 1 : Using cached data (hash :                           ]8;id=5599399;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599400;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_1044820769_2793713394)                                                     

[05/18/26 14:09:42] INFO     Animation 2 : Using cached data (hash :                           ]8;id=5599405;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599406;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_2514011427_423103456)                                                      

                    INFO     Animation 3 : Using cached data (hash :                           ]8;id=5599411;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599412;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_58530935_2725641452)                                                       

...

                    INFO     Animation 13 : Using cached data (hash :                          ]8;id=5599471;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599472;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_475911201_3605453475)                                                      

[05/18/26 14:09:47] INFO     Animation 14 : Using cached data (hash :                          ]8;id=5599477;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599478;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_952528981_1274200292)                                                      

                    INFO     Combining to Movie file.                                      ]8;id=5599485;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5599486;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#753\753]8;;\

                    INFO                                                                   ]8;id=5599492;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5599493;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#904\904]8;;\
                             File ready at                                                                         
                             '/workspaces/OCR-A-Level-Computing-Lessons/media/videos/OCR-A                         
                             -Level-Computing-Lessons/720p30/SignMagnitudeDemo.mp4'                                
                                                                                                                   

                    INFO     Rendered SignMagnitudeDemo                                                ]8;id=5599500;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=5599501;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene.py#278\278]8;;\
                             Played 15 animations                                                                  

In [7]:
%%manim -qm TwosComplementConversionDemo

from manim import *


class TwosComplementConversionDemo(Scene):

    def construct(self):
        title = Text("Two's Complement: Finding -77", font_size=40).to_edge(UP)
        self.play(Write(title))

        s2 = VGroup(
            Text("Find -77 in two's complement", font_size=30, color=GREEN),
            Text("Step 1: Write +77 in binary", font_size=28),
            Text("+77  =  01001101", font_size=30),
            Text("Step 2: Flip all bits (one's complement)", font_size=28),
            Text("Flip  ->  10110010", font_size=30),
            Text("Step 3: Add 1", font_size=28),
            Text("10110010 + 1  =  10110011  =  \u221277", font_size=30, color=YELLOW),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.3).next_to(title, DOWN, buff=0.45).to_edge(LEFT, buff=0.8)

        for line in s2:
            self.play(FadeIn(line, shift=RIGHT * 0.15), run_time=0.6)
            self.wait(0.2)

        final_box = SurroundingRectangle(s2[-1], color=GREEN, buff=0.15)
        self.play(Create(final_box))
        self.wait(20)


Manim Community v0.20.1

[05/18/26 14:09:48] INFO     Animation 0 : Using cached data (hash :                           ]8;id=5599506;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599507;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             1987512680_2268936007_223132457)                                                      

                    INFO     Animation 1 : Using cached data (hash :                           ]8;id=5599512;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599513;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_811071581_1078249560)                                                      

                    INFO     Animation 2 : Using cached data (hash :                           ]8;id=5599518;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599519;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_2514011427_2342766072)                                                     

[05/18/26 14:09:49] INFO     Animation 3 : Using cached data (hash :                           ]8;id=5599524;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599525;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_432003383_600990008)                                                       

...

[05/18/26 14:09:55] INFO     Animation 15 : Using cached data (hash :                          ]8;id=5599596;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599597;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_3292625411_323991843)                                                      

[05/18/26 14:09:56] INFO     Animation 16 : Using cached data (hash :                          ]8;id=5599602;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599603;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_952528981_4164419739)                                                      

                    INFO     Combining to Movie file.                                      ]8;id=5599608;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5599609;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#753\753]8;;\

                    INFO                                                                   ]8;id=5599614;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5599615;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#904\904]8;;\
                             File ready at                                                                         
                             '/workspaces/OCR-A-Level-Computing-Lessons/media/videos/OCR-A                         
                             -Level-Computing-Lessons/720p30/TwosComplementConversionDemo.                         
                             mp4'                                                                                  
                                                                                                                   

                    INFO     Rendered TwosComplementConversionDemo                                     ]8;id=5599620;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=5599621;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene.py#278\278]8;;\
                             Played 17 animations                                                                  

In [8]:
%%manim -qm BinarySubtractionDemo

from manim import *


class BinarySubtractionDemo(Scene):

    def construct(self):
        title = Text("Binary Subtraction: 65 \u2212 43", font_size=40).to_edge(UP)
        self.play(Write(title))

        head = Text("Using two's complement addition", font_size=30, color=BLUE).next_to(title, DOWN, buff=0.5)
        self.play(FadeIn(head))

        eq1 = Text("65       =  01000001", font_size=32)
        eq2 = Text("\u221243      =  11010101", font_size=32)
        eq3 = Text("Sum      =  1 00010110", font_size=32)
        eq4 = Text("Ignore carry  \u2192  00010110  =  22", font_size=32, color=GREEN)
        work = VGroup(eq1, eq2, eq3, eq4).arrange(DOWN, aligned_edge=LEFT, buff=0.35).next_to(head, DOWN, buff=0.4)

        for eq in (eq1, eq2, eq3):
            self.play(FadeIn(eq, shift=UP * 0.1))

        carry_mark = SurroundingRectangle(eq3[11], color=RED, buff=0.05)
        self.play(Create(carry_mark), run_time=0.4)
        self.play(FadeIn(eq4, shift=UP * 0.1))

        final_box = SurroundingRectangle(eq4, color=GREEN, buff=0.15)
        self.play(Create(final_box), run_time=0.5)
        self.wait(20)


Manim Community v0.20.1

                    INFO     Animation 0 : Using cached data (hash :                           ]8;id=5599626;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599627;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             1987512680_2713059993_223132457)                                                      

                    INFO     Animation 1 : Using cached data (hash :                           ]8;id=5599632;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599633;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_3160911556_1434555680)                                                     

[05/18/26 14:09:57] INFO     Animation 2 : Using cached data (hash :                           ]8;id=5599638;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599639;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_97299597_100514000)                                                        

                    INFO     Animation 3 : Using cached data (hash :                           ]8;id=5599644;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599645;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_1161247532_3951839316)                                                     

...

                    INFO     Animation 7 : Using cached data (hash :                           ]8;id=5599668;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599669;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_2047523991_836847766)                                                      

[05/18/26 14:10:05] INFO     Animation 8 : Partial movie file written in                   ]8;id=5599675;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5599676;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/workspaces/OCR-A-Level-Computing-Lessons/media/videos/OCR-A                         
                             -Level-Computing-Lessons/720p30/partial_movie_files/BinarySub                         
                             tractionDemo/2538612922_1050081891_1181395200.mp4'                                    

                    INFO     Combining to Movie file.                                      ]8;id=5599681;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5599682;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#753\753]8;;\

                    INFO                                                                   ]8;id=5599687;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5599688;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#904\904]8;;\
                             File ready at                                                                         
                             '/workspaces/OCR-A-Level-Computing-Lessons/media/videos/OCR-A                         
                             -Level-Computing-Lessons/720p30/BinarySubtractionDemo.mp4'                            
                                                                                                                   

                    INFO     Rendered BinarySubtractionDemo                                            ]8;id=5599693;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=5599694;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene.py#278\278]8;;\
                             Played 9 animations                                                                   

In [9]:
# Stitch the three signed binary clips into one video using ffmpeg (bundled with Manim).
# Run this cell AFTER all three animation cells above have been executed.
import subprocess
from pathlib import Path

vid_dir = Path.cwd() / "media" / "videos" / Path.cwd().resolve().name / "720p30"
clips = [
    vid_dir / "SignMagnitudeDemo.mp4",
    vid_dir / "TwosComplementConversionDemo.mp4",
    vid_dir / "BinarySubtractionDemo.mp4",
]
output = vid_dir / "SignedBinaryWorkedSteps_Full.mp4"

missing = [c for c in clips if not c.exists()]
if missing:
    print("Run the animation cells above first. Missing:", [c.name for c in missing])
else:
    list_file = vid_dir / "_signed_binary_concat.txt"
    list_file.write_text("\n".join(f"file '{c}'" for c in clips), encoding="utf-8")
    proc = subprocess.run(
        ["ffmpeg", "-y", "-f", "concat", "-safe", "0",
         "-i", str(list_file), "-c", "copy", str(output)],
        capture_output=True, text=True,
    )
    if proc.returncode == 0:
        print("Full video saved to:", output.name)
    else:
        print("ffmpeg error:", proc.stderr[-600:])


Full video saved to: SignedBinaryWorkedSteps_Full.mp4


## Floating point representation (teaching model)
In this lesson model, a floating point value is shown as:
- Mantissa (with sign), then
- Exponent (with sign)

Example format used in the slides: mantissa 8 bits + exponent 4 bits.

Key idea for addition/subtraction:
- Equalise exponents first (align binary points),
- then add/subtract mantissas,
- then renormalise.

In [10]:
# Decode teaching-format floating point values from the slides
def signed_from_bits(bits: str) -> int:
    n = len(bits)
    value = int(bits, 2)
    if bits[0] == '1':
        value -= 1 << n
    return value

def mantissa_fraction(m_bits: str) -> float:
    sign = -1 if m_bits[0] == '1' else 1
    frac_bits = m_bits[1:]
    frac = 0.0
    for i, bit in enumerate(frac_bits, start=1):
        if bit == '1':
            frac += 2 ** (-i)
    return sign * frac

def decode_fp(m_bits: str, e_bits: str) -> float:
    m = mantissa_fraction(m_bits)
    e = signed_from_bits(e_bits)
    return m * (2 ** e)

examples = [
    ("0.1101100", "0100"),
    ("1.0010100", "0011"),
    ("0.1100000", "1110"),
]

for m, e in examples:
    value = decode_fp(m.replace('.', ''), e)
    print(f"{m} {e} -> {value}")

0.1101100 0100 -> 13.5
1.0010100 0011 -> -1.25
0.1100000 1110 -> 0.1875


In [11]:
# ─── OCR A Level answer-verification assertions ───────────────────────────────
# Run this cell to confirm all helper functions match the OCR mark scheme.

# 1. Sign and magnitude: +3 and -3 in 8 bits
assert sign_magnitude( 3, 8) == "00000011"
assert sign_magnitude(-3, 8) == "10000011"

# Adding them gives the wrong answer — that is the whole lesson point
_s3p = int(sign_magnitude( 3, 8), 2)
_s3n = int(sign_magnitude(-3, 8), 2)
_sm_sum = to_bin(_s3p + _s3n, 8)
assert _sm_sum == "10000110", f"SM add: expected 10000110, got {_sm_sum}"
assert _sm_sum != "00000000", "SM +3 + -3 must NOT give 0"

# 2. Two's complement conversion of -77: flip bits of +77, then add 1
assert twos_complement( 77, 8) == "01001101"
_ones_77 = "".join("0" if b == "1" else "1" for b in "01001101")
assert _ones_77 == "10110010"
assert twos_complement(-77, 8) == "10110011"

# 3. Subtraction 65 − 43 = 22 via two's complement addition
assert twos_complement( 65, 8) == "01000001"
assert twos_complement(-43, 8) == "11010101"
_raw = int("01000001", 2) + int("11010101", 2)   # 65 + 213 = 278 (9 bits)
assert _raw == 278
assert to_bin(_raw, 8) == "00010110"             # drop carry → 22
assert int("00010110", 2) == 22

# 4. Floating-point decode: 0.1100000 × 2^1 = 1.5  and  × 2^-2 = 0.1875
assert abs(decode_fp("01100000", "0001") - 1.5)    < 1e-9
assert abs(decode_fp("01100000", "1110") - 0.1875) < 1e-9   # "1110" = −2 in 4-bit TC

# 5. Normalisation rule: positive starts "01", negative starts "10"
def _is_norm(m: str) -> bool:
    return m[:2] in ("01", "10")

assert     _is_norm("01100000")
assert     _is_norm("10011000")
assert not _is_norm("00100000")
assert not _is_norm("11010000")

print("✓ All OCR A Level answer-verification checks passed")


✓ All OCR A Level answer-verification checks passed


In [12]:
%%manim -qm FloatingPointAdditionWorkedSteps

from manim import *



class FloatingPointAdditionWorkedSteps(Scene):

    def construct(self):

        title = Text("Floating Point Addition: Step by Step", font_size=40).to_edge(UP)

        self.play(Write(title))



        problem = Text("A = 0.1100000 0001,   B = 0.1111100 0011", font_size=30).shift(UP * 2.0)

        self.play(FadeIn(problem))



        steps = [

            "1) Convert to fixed point by applying each exponent",

            "   A -> 1.1000      B -> 111.1100",

            "2) Add mantissas in fixed point",

            "   1.1000 + 111.1100 = 1001.0100",

            "3) Number is positive, so sign bit = 0",

            "4) Renormalise result",

            "   1001.0100 -> 0.1001010 x 2^4",

            "5) Final normalised floating point",

            "   Result = 0.1001010 0100",

        ]



        lines = VGroup(*[Text(s, font_size=28) for s in steps]).arrange(DOWN, aligned_edge=LEFT, buff=0.28)

        lines.next_to(problem, DOWN, buff=0.45).to_edge(LEFT, buff=0.7)



        for i, line in enumerate(lines):

            self.play(FadeIn(line, shift=UP * 0.15), run_time=0.55)

            if i in (1, 3, 6, 8):

                box = SurroundingRectangle(line, color=YELLOW, buff=0.1)

                self.play(Create(box), run_time=0.35)

                self.play(FadeOut(box), run_time=0.25)

            self.wait(0.2)



        final_box = SurroundingRectangle(lines[-1], color=GREEN, buff=0.15)

        self.play(Create(final_box))

        self.wait(20)

Manim Community v0.20.1

[05/18/26 14:10:06] INFO     Animation 0 : Using cached data (hash :                           ]8;id=5599699;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599700;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             1987512680_3474714760_223132457)                                                      

                    INFO     Animation 1 : Using cached data (hash :                           ]8;id=5599705;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599706;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_1264038258_1707633780)                                                     

[05/18/26 14:10:07] INFO     Animation 2 : Using cached data (hash :                           ]8;id=5599711;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599712;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_1949749973_3833288471)                                                     

[05/18/26 14:10:08] INFO     Animation 3 : Using cached data (hash :                           ]8;id=5599717;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599718;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_2514011427_2897856193)                                                     

...

[05/18/26 14:10:30] INFO     Animation 28 : Using cached data (hash :                          ]8;id=5599867;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599868;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_2501375178_898767456)                                                      

[05/18/26 14:10:31] INFO     Animation 29 : Using cached data (hash :                          ]8;id=5599873;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599874;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_1050081891_3622197785)                                                     

                    INFO     Combining to Movie file.                                      ]8;id=5599879;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5599880;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#753\753]8;;\

                    INFO                                                                   ]8;id=5599885;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5599886;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#904\904]8;;\
                             File ready at                                                                         
                             '/workspaces/OCR-A-Level-Computing-Lessons/media/videos/OCR-A                         
                             -Level-Computing-Lessons/720p30/FloatingPointAdditionWorkedSt                         
                             eps.mp4'                                                                              
                                                                                                                   

                    INFO     Rendered FloatingPointAdditionWorkedSteps                                 ]8;id=5599891;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=5599892;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene.py#278\278]8;;\
                             Played 30 animations                                                                  

## Student Practice Space: Random Floating-Point Questions

Use this section for live class questions before revealing the subtraction walkthrough.

Suggested prompts to generate on the fly:
- Convert a floating-point value to denary.
- Normalise an unnormalised mantissa/exponent pair.
- Add two floating-point numbers after converting to fixed point.

Pause here for students to attempt, then run the next worked-example cell.

In [13]:
import random

random.seed()   # fresh seed each run

def _rand_mantissa(neg=False, bits=8):
    """Normalised mantissa; last bit fixed to 0 so a right-shift is lossless."""
    pfx = "10" if neg else "01"
    mid = "".join(random.choice("01") for _ in range(bits - 3))
    return pfx + mid + "0"

def _rand_exp4(lo=-5, hi=5):
    e = random.randint(lo, hi)
    return format(e & 0xF, "04b"), e

N = 3  # questions per type

print("══ Type 1: Decode floating-point to denary ══")
for _ in range(N):
    m = _rand_mantissa(neg=random.choice([False, True]))
    e_bits, e_val = _rand_exp4()
    val = decode_fp(m, e_bits)
    print(f"  {m[0]}.{m[1:]}  exp {e_bits} ({e_val:+d})  →  {val:.6g}")

print()
print("══ Type 2: Two's complement subtraction ══")
for _ in range(N):
    a = random.randint(50, 120)
    b = random.randint(10, min(a - 1, 127))
    a_tc   = twos_complement(a,  8)
    neg_tc = twos_complement(-b, 8)
    raw    = int(a_tc, 2) + int(neg_tc, 2)
    result = int(to_bin(raw, 8), 2)
    assert result == a - b, f"{a}−{b}: expected {a-b}, got {result}"
    print(f"  {a} − {b} = ?   [{a}: {a_tc}, −{b}: {neg_tc}]   → {a-b} = {to_bin(raw, 8)}")

print()
print("══ Type 3: Normalise the following floating-point value ══")
for _ in range(N):
    neg = random.choice([False, True])
    m_norm = _rand_mantissa(neg=neg)
    _, e_norm = _rand_exp4(-4, 4)
    # Shift mantissa right by 1 to unnormalise (prepend sign bit, drop last bit which is 0)
    m_unnorm = m_norm[0] + m_norm[:-1]
    e_unnorm = e_norm + 1
    assert abs(decode_fp(m_unnorm, format(e_unnorm & 0xF, "04b")) -
               decode_fp(m_norm,   format(e_norm   & 0xF, "04b"))) < 1e-9, "value changed"
    print(f"  Unnorm: {m_unnorm[0]}.{m_unnorm[1:]}  exp {format(e_unnorm&0xF,'04b')} ({e_unnorm:+d})")
    print(f"  → Norm: {m_norm[0]}.{m_norm[1:]}  exp {format(e_norm&0xF,'04b')} ({e_norm:+d})"
          f"  [value: {decode_fp(m_norm, format(e_norm&0xF,'04b')):.6g}]")
    print()


══ Type 1: Decode floating-point to denary ══
  0.1011010  exp 0000 (+0)  →  0.703125
  0.1010110  exp 0001 (+1)  →  1.34375
  1.0110110  exp 1101 (-3)  →  -0.0527344

══ Type 2: Two's complement subtraction ══
  74 − 35 = ?   [74: 01001010, −35: 11011101]   → 39 = 00100111
  54 − 20 = ?   [54: 00110110, −20: 11101100]   → 34 = 00100010
  103 − 92 = ?   [103: 01100111, −92: 10100100]   → 11 = 00001011

══ Type 3: Normalise the following floating-point value ══
  Unnorm: 0.0101110  exp 0010 (+2)
  → Norm: 0.1011100  exp 0001 (+1)  [value: 1.4375]

  Unnorm: 0.0111101  exp 0000 (+0)
  → Norm: 0.1111010  exp 1111 (-1)  [value: 0.476562]

  Unnorm: 0.0101101  exp 0000 (+0)
  → Norm: 0.1011010  exp 1111 (-1)  [value: 0.351562]



In [14]:
%%manim -qm FloatingPointSubtractionWorkedSteps

from manim import *



class FloatingPointSubtractionWorkedSteps(Scene):

    def construct(self):

        title = Text("Floating Point Subtraction: Step by Step", font_size=40).to_edge(UP)

        self.play(Write(title))



        problem = Text("A = 0.1101000 0100,   B = 0.1110000 0011", font_size=30).shift(UP * 2.0)

        self.play(FadeIn(problem))



        steps = [

            "1) Convert each value to fixed point",

            "   A -> 1101.0000     B -> 0111.0000",

            "2) Subtract by adding negative B",

            "   One's complement(B) -> 1000.1111",

            "   Two's complement(B) -> 1001.0000",

            "3) Add A + (-B)",

            "   1101.0000 + 1001.0000 = 0110.0000 (ignore overflow)",

            "4) Convert back to normalised floating point",

            "   Result = 0.1100000 0011",

        ]



        lines = VGroup(*[Text(s, font_size=28) for s in steps]).arrange(DOWN, aligned_edge=LEFT, buff=0.26)

        lines.next_to(problem, DOWN, buff=0.45).to_edge(LEFT, buff=0.7)



        for i, line in enumerate(lines):

            self.play(FadeIn(line, shift=UP * 0.15), run_time=0.55)

            if i in (1, 3, 4, 6, 8):

                box = SurroundingRectangle(line, color=YELLOW, buff=0.1)

                self.play(Create(box), run_time=0.35)

                self.play(FadeOut(box), run_time=0.25)

            self.wait(0.2)



        final_box = SurroundingRectangle(lines[-1], color=GREEN, buff=0.15)

        self.play(Create(final_box))

        self.wait(20)

Manim Community v0.20.1

[05/18/26 14:10:32] INFO     Animation 0 : Using cached data (hash :                           ]8;id=5599897;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599898;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             1987512680_1985996451_223132457)                                                      

                    INFO     Animation 1 : Using cached data (hash :                           ]8;id=5599903;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599904;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_523418604_3027475952)                                                      

[05/18/26 14:10:33] INFO     Animation 2 : Using cached data (hash :                           ]8;id=5599909;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5599910;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_3762390400_2345337461)                                                     

[05/18/26 14:10:34] INFO     Animation 3 : Partial movie file written in                   ]8;id=5599915;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5599916;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/workspaces/OCR-A-Level-Computing-Lessons/media/videos/OCR-A                         
                             -Level-Computing-Lessons/720p30/partial_movie_files/FloatingP                         
                             ointSubtractionWorkedSteps/2538612922_3142935987_1126555814.m                         
                             p4'                                                                                   

...

[05/18/26 14:10:57] INFO     Animation 30 : Using cached data (hash :                          ]8;id=5600077;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=5600078;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_4129695861_2447792808)                                                     

[05/18/26 14:11:05] INFO     Animation 31 : Partial movie file written in                  ]8;id=5600083;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5600084;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/workspaces/OCR-A-Level-Computing-Lessons/media/videos/OCR-A                         
                             -Level-Computing-Lessons/720p30/partial_movie_files/FloatingP                         
                             ointSubtractionWorkedSteps/2538612922_952528981_2547481360.mp                         
                             4'                                                                                    

                    INFO     Combining to Movie file.                                      ]8;id=5600089;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5600090;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#753\753]8;;\

                    INFO                                                                   ]8;id=5600095;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=5600096;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#904\904]8;;\
                             File ready at                                                                         
                             '/workspaces/OCR-A-Level-Computing-Lessons/media/videos/OCR-A                         
                             -Level-Computing-Lessons/720p30/FloatingPointSubtractionWorke                         
                             dSteps.mp4'                                                                           
                                                                                                                   

                    INFO     Rendered FloatingPointSubtractionWorkedSteps                              ]8;id=5600101;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=5600102;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene.py#278\278]8;;\
                             Played 32 animations                                                                  

## Plenary quick-check
1. Why is two's complement preferred over sign and magnitude in processors?
2. In 8-bit two's complement, what is the range of values?
3. What does overflow mean in fixed-width binary arithmetic?
4. When adding floating point numbers, why must exponents be aligned first?
5. What is the normalised pattern after the binary point for:
   - positive mantissas?
   - negative mantissas?

You can now extend this notebook with your worksheet questions as practice slides.